# sample2 - llama-cpp-python で Python から制御

**llama-cpp-python** を使って GGUF 形式の量子化モデルを Python から直接制御します。  
Ollama より低レベルな制御が可能で、パラメータの細かい調整ができます。

| ステップ | 内容 |
|----------|------|
| 1 | モデルのダウンロード |
| 2 | モデルの読み込み |
| 3 | テキスト生成 |
| 4 | チャット形式 |
| 5 | 生成パラメータの調整 |

## 事前準備

GGUF モデルを Hugging Face からダウンロードします。  
ターミナルで以下を実行してください（約4GB）。

```bash
pip install huggingface-hub
hf --help
hf download bartowski/Meta-Llama-3-8B-Instruct-GGUF \
  Meta-Llama-3-8B-Instruct-Q4_K_M.gguf \
  --local-dir .
```

## 1. モデルの読み込み

In [4]:
from llama_cpp import Llama

# モデルの読み込み
# n_gpu_layers=-1 で全レイヤーを GPU に載せる（GPU版のみ）
# n_gpu_layers=0  で CPU のみ使用
llm = Llama(
    model_path='./Meta-Llama-3-8B-Instruct-Q4_K_M.gguf',
    n_ctx=2048,        # コンテキスト長
    n_gpu_layers=0,    # CPU で実行（GPU版は -1）
    verbose=False
)
print("モデル読み込み完了")

llama_context: n_ctx_per_seq (2048) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


モデル読み込み完了


## 2. テキスト生成

In [5]:
# シンプルなテキスト補完
output = llm(
    "日本の首都は",
    max_tokens=50,
    echo=True  # プロンプトも含めて出力
)

print(output['choices'][0]['text'])
print("\n--- トークン情報 ---")
print("プロンプトトークン:", output['usage']['prompt_tokens'])
print("生成トークン      :", output['usage']['completion_tokens'])

日本の首都は Tokyo（東京都）です。. The Japanese government officially recognizes Tokyo as the capital of Japan, and it is the country's largest city in terms of population. Tokyo is located on the eastern coast of Honshu, the largest island of Japan

--- トークン情報 ---
プロンプトトークン: 6
生成トークン      : 50


## 3. チャット形式（OpenAI 互換 API）

llama-cpp-python は OpenAI 互換の API を提供しています。

In [6]:
response = llm.create_chat_completion(
    messages=[
        {'role': 'system',  'content': 'あなたは親切な日本語アシスタントです。'},
        {'role': 'user',    'content': 'PyTorch を一言で説明してください。'}
    ]
)

print("応答:")
print(response['choices'][0]['message']['content'])

応答:
PyTorchは、Pythonで書かれたオープンソースの機械学習フレームワークです。Deep LearningやNatural Language Processing、Computer Visionなどの分野で活躍しています。PyTorchは、Tensor計算を中心とした設計で、モデルを定義し、トレーニングや推論を行うことができます。Pythonのインタプリターに近い設計のため、実験や開発が速く、柔軟に対応することができます。


## 4. ストリーミング出力

In [7]:
print("応答（ストリーミング）:")
for chunk in llm.create_chat_completion(
    messages=[{'role': 'user', 'content': '機械学習の学習ステップを3つ挙げてください。'}],
    stream=True
):
    delta = chunk['choices'][0]['delta']
    if 'content' in delta:
        print(delta['content'], end='', flush=True)
print()

応答（ストリーミング）:
Here are three common steps in the machine learning learning process:

1. **Data Collection and Preprocessing**: This step involves gathering relevant data, cleaning and preprocessing it, and transforming it into a format that can be used for training a machine learning model. This may include tasks such as data cleaning, feature scaling, and normalization.
2. **Model Training**: In this step, a machine learning algorithm is trained on the preprocessed data to learn patterns and relationships. The algorithm is typically trained using a subset of the data, known as the training set, and the goal is to minimize the error or loss function.
3. **Model Evaluation and Deployment**: In this final step, the trained model is evaluated on a separate test set to assess its performance and accuracy. If the model performs well, it can be deployed in a production environment to make predictions or take actions. This may involve fine-tuning the model, selecting the best hyperparameters, 

## 5. 生成パラメータの調整

| パラメータ | 説明 | 値の範囲 |
|-----------|------|----------|
| `temperature` | 出力のランダム性。高いほど創造的 | 0.0〜2.0 |
| `top_p` | 確率の上位 p% のトークンから選択 | 0.0〜1.0 |
| `max_tokens` | 最大生成トークン数 | 1〜 |
| `repeat_penalty` | 繰り返しへのペナルティ | 1.0〜 |

In [8]:
prompt = [{'role': 'user', 'content': '空が青い理由を一文で説明してください。'}]

# temperature を変えて比較
for temp in [0.1, 0.7, 1.5]:
    response = llm.create_chat_completion(
        messages=prompt,
        temperature=temp,
        max_tokens=100
    )
    print(f"temperature={temp}:")
    print(response['choices'][0]['message']['content'])
    print()

temperature=0.1:
A simple yet profound question! 😊

The reason why the sky appears blue is because of a phenomenon called Rayleigh scattering, in which shorter (blue) wavelengths of light are scattered more than longer (red) wavelengths by the tiny molecules of gases in the atmosphere, such as nitrogen and oxygen. This scattering effect gives the sky its blue color, making it appear blue to our eyes.

temperature=0.7:
Here's a one-sentence explanation:

The sky appears blue because of a phenomenon called Rayleigh scattering, in which shorter wavelengths of light (such as blue and violet) are scattered more efficiently by the tiny molecules of gases in the atmosphere, while longer wavelengths (such as red and orange) continue to travel in a more direct path to our eyes, giving the sky its blue appearance.

temperature=1.5:
A classic question! 😊

The reason why the sky appears blue is due to a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh. He disc

# ダウンロードしたモデルを削除

In [1]:
import os

# GGUF モデルファイルの削除
gguf_file = './Meta-Llama-3-8B-Instruct-Q4_K_M.gguf'
if os.path.exists(gguf_file):
    os.remove(gguf_file)
    print(f"削除: {gguf_file}")
else:
    print(f"なし: {gguf_file}")

削除: ./Meta-Llama-3-8B-Instruct-Q4_K_M.gguf
